# Task 5: Smart Data Analysis Platform
### BetaBytez AI/ML Capstone — Umme Habiba Malik

## Day 1: Core Engine Prototype

Building the code-generation + execution engine that powers both:
- Automated EDA (on CSV upload)
- Natural language Q&A on the dataset

**Architecture:** Groq LLM generates pandas/matplotlib code → `exec()` runs it → errors get fed back to the LLM for a retry (capped at N attempts).

**Plan for today:**
1. Setup & dependencies
2. CSV upload + dataframe profiling (the `df_info` context we feed the LLM)
3. Code-gen agent (`generate_code`)
4. Safe execution sandbox (`safe_exec`)
5. Retry loop (`run_with_retry`)
6. Auto-EDA built on top of the engine
7. Natural language Q&A built on top of the engine
8. Test with 2–3 sample CSVs (clean, messy/nulls, categorical-heavy)

## 1. Setup & Dependencies

In [ ]:
# Install required packages (quiet mode to keep output clean)
!pip install groq pandas matplotlib seaborn -q

In [ ]:
# Core imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import contextlib
import traceback
from groq import Groq

# Keep plots looking consistent across the notebook
sns.set_theme(style="whitegrid")

### Groq API key setup (secure)

We do **not** hardcode the API key in the notebook — this repo gets pushed to GitHub,
and a leaked key in a `.ipynb` is just as bad as a leaked `.env` file.

Steps:
1. Click the 🔑 key icon in the Colab left sidebar
2. Add a new secret named `GROQ_API_KEY`
3. Paste your key as the value, toggle **Notebook access** on

In [ ]:
# Load the API key from Colab Secrets (never printed, never hardcoded)
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# Model used across the engine — fast + free-tier friendly
MODEL_NAME = "llama-3.1-8b-instant"

In [ ]:
# Sanity check: confirm the Groq client + key actually work before building anything on top
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Say 'setup working' and nothing else."}]
)
print(response.choices[0].message.content)

## 2. CSV Upload + Dataframe Profiling

This gives us the `df_info` context string that we hand to the LLM in every prompt,
so it knows column names, dtypes, and a data sample without us pasting the whole CSV.

In [ ]:
# Upload a CSV file (Colab file picker)
from google.colab import files

uploaded = files.upload()  # opens a picker; select your CSV
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)

print(f"Loaded '{csv_filename}' — shape: {df.shape}")
df.head()

In [ ]:
def get_df_info(df: pd.DataFrame) -> str:
    """
    Build a compact text summary of the dataframe to feed into LLM prompts.
    Includes shape, column dtypes, null counts, and a small sample —
    enough context for the LLM to write correct pandas code without
    us sending the entire dataset (wastes tokens, risks leaking full data).
    """
    buffer = io.StringIO()
    df.info(buf=buffer)
    info_str = buffer.getvalue()

    null_counts = df.isnull().sum()
    nulls_str = null_counts[null_counts > 0].to_string() if null_counts.sum() > 0 else "No missing values"

    summary = f"""
Shape: {df.shape[0]} rows, {df.shape[1]} columns

Column info:
{info_str}

Null counts (columns with nulls only):
{nulls_str}

Sample rows:
{df.head(3).to_string()}
"""
    return summary.strip()


# Test it
df_info = get_df_info(df)
print(df_info)

## 3. Code-Gen Agent

Takes a task description (either a fixed EDA prompt or a user's natural language
question) plus the `df_info` context, and asks Groq to generate **only** executable
Python code — no explanations, no markdown fences, so we can `exec()` it directly.

In [ ]:
SYSTEM_PROMPT = """You are a Python data analysis code generator.

Rules:
- You will be given info about a pandas DataFrame called `df` (already loaded, do not reload or recreate it).
- Write Python code that accomplishes the user's task using `df`.
- Use pandas, matplotlib.pyplot (as plt), and seaborn (as sns) — all already imported.
- If creating a chart, call plt.show() at the end of that chart's code.
- If the task expects a text/numeric result, assign it to a variable named `result` and print it.
- Return ONLY raw Python code. No markdown fences (no ```), no explanations, no comments about what you're doing outside the code.
- Keep code safe: no file I/O, no network calls, no imports beyond what's already available.
"""


def generate_code(task_description: str, df_info: str, error_context: str = None) -> str:
    """
    Call Groq to generate pandas/matplotlib code for a given task.

    If error_context is provided, this is a retry — we include the previous
    code's error so the LLM can fix its mistake instead of guessing blind.
    """
    user_prompt = f"""DataFrame info:
{df_info}

Task: {task_description}
"""

    if error_context:
        user_prompt += f"""
IMPORTANT: Your previous attempt failed with this error:
{error_context}

Fix the code so it runs without error. Return the corrected code only.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,  # low temperature: we want reliable code, not creative code
    )

    code = response.choices[0].message.content.strip()

    # Safety net: strip markdown fences if the model adds them anyway
    if code.startswith("```"):
        code = code.strip("`")
        if code.startswith("python"):
            code = code[len("python"):]
    return code.strip()

## 4. Safe Execution Sandbox

Runs generated code in a restricted namespace (only `df`, pandas, matplotlib, seaborn
available — no builtins like `open`, `eval`, `__import__`). Captures stdout, any
figures created, and the `result` variable if the code set one.

In [ ]:
def safe_exec(code: str, df: pd.DataFrame) -> dict:
    """
    Execute LLM-generated code in a restricted namespace.

    Returns a dict with:
      - success: bool
      - stdout: captured print() output
      - result: value of `result` variable, if set by the code
      - figures: list of matplotlib figures created during execution
      - error: traceback string, if execution failed
    """
    # Restricted globals: only safe, needed objects are exposed to the exec'd code
    safe_globals = {
        "__builtins__": {
            "print": print, "len": len, "range": range, "str": str, "int": int,
            "float": float, "list": list, "dict": dict, "set": set, "tuple": tuple,
            "sum": sum, "min": min, "max": max, "sorted": sorted, "enumerate": enumerate,
            "zip": zip, "round": round, "abs": abs, "bool": bool,
        },
        "pd": pd,
        "plt": plt,
        "sns": sns,
        "df": df,
    }
    local_vars = {}
    stdout_capture = io.StringIO()

    plt.close("all")  # clear any stale figures before running

    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(code, safe_globals, local_vars)

        figures = [plt.figure(n) for n in plt.get_fignums()]

        return {
            "success": True,
            "stdout": stdout_capture.getvalue(),
            "result": local_vars.get("result"),
            "figures": figures,
            "error": None,
        }
    except Exception:
        return {
            "success": False,
            "stdout": stdout_capture.getvalue(),
            "result": None,
            "figures": [],
            "error": traceback.format_exc(),
        }

## 5. Retry Loop

Ties `generate_code` and `safe_exec` together: generate → run → if it fails,
send the error back to the LLM and try again, up to `max_retries` times.

In [ ]:
def run_with_retry(task_description: str, df: pd.DataFrame, df_info: str, max_retries: int = 3, verbose: bool = True) -> dict:
    """
    Generate + execute code for a task, retrying on failure.

    Each retry feeds the previous error back to the LLM so it can self-correct.
    Returns the last execution result dict, plus 'attempts' and 'final_code'
    so we can show/log what actually ran.
    """
    error_context = None
    last_code = None
    last_exec_result = None

    for attempt in range(1, max_retries + 1):
        if verbose:
            print(f"--- Attempt {attempt}/{max_retries} ---")

        code = generate_code(task_description, df_info, error_context)
        last_code = code

        exec_result = safe_exec(code, df)
        last_exec_result = exec_result

        if exec_result["success"]:
            if verbose:
                print("Succeeded.")
            exec_result["attempts"] = attempt
            exec_result["final_code"] = code
            return exec_result

        # Failed — prep the error as context for the next attempt
        error_context = exec_result["error"]
        if verbose:
            print(f"Failed: {error_context.splitlines()[-1]}")

    # All retries exhausted — return the last failure so the caller can show it
    last_exec_result["attempts"] = max_retries
    last_exec_result["final_code"] = last_code
    return last_exec_result

## 6. Automated EDA (built on the engine)

A fixed task description that asks for standard profiling: dtypes, nulls,
describe(), and a handful of useful charts. Reuses `run_with_retry` — no new
execution logic needed.

In [ ]:
EDA_TASK = """Perform exploratory data analysis on df:
1. Print df.dtypes
2. Print df.isnull().sum()
3. Print df.describe() for numeric columns
4. Create a histogram for each numeric column (use subplots if there are multiple)
5. If there are categorical columns with few unique values, create a bar chart of value counts for up to 2 of them
Assign a short text summary (2-3 sentences) of key observations to a variable named `result`.
"""

eda_output = run_with_retry(EDA_TASK, df, df_info, max_retries=3)

print("\n=== STDOUT ===")
print(eda_output["stdout"])
print("\n=== RESULT SUMMARY ===")
print(eda_output["result"])
print(f"\n(Succeeded on attempt {eda_output['attempts']})")

## 7. Natural Language Q&A (built on the engine)

Same engine, different task description — this time the user's own question.

In [ ]:
def ask_question(question: str, df: pd.DataFrame, df_info: str, max_retries: int = 3) -> dict:
    """
    Answer a natural language question about df using the codegen + retry engine.
    Wraps the raw question with a small instruction so the LLM knows to set `result`.
    """
    task = f"{question}\nIf the answer is a number/text, assign it to `result` and print it. If it's better shown as a chart, create the chart."
    return run_with_retry(task, df, df_info, max_retries=max_retries)


# Example usage — replace with a question relevant to your uploaded CSV
qa_output = ask_question("What is the average value of the first numeric column?", df, df_info)

print("\n=== STDOUT ===")
print(qa_output["stdout"])
print("\n=== RESULT ===")
print(qa_output["result"])
print(f"\n(Succeeded on attempt {qa_output['attempts']})")

## 8. Stress-Testing the Retry Loop

The retry loop is the highest-risk part of this system — it needs to work on
messy, real-world data, not just a clean sample. Test with at least 2–3 CSVs:
- A clean, mostly-numeric dataset
- A messy dataset with nulls / mixed types
- A categorical-heavy dataset

Re-run cells in Section 2 (upload) through Section 7 (Q&A) for each test CSV,
and note below whether the retry loop recovered from any failures.

In [ ]:
# Quick helper to log test runs across multiple CSVs — useful for your report/demo
test_log = []

def log_test(csv_name: str, task: str, exec_result: dict):
    test_log.append({
        "csv": csv_name,
        "task": task,
        "success": exec_result["success"],
        "attempts": exec_result["attempts"],
    })

# Example: log_test(csv_filename, EDA_TASK, eda_output)
# After testing a few CSVs, view the log:
pd.DataFrame(test_log)

## Next steps (Day 2)
- Port `generate_code`, `safe_exec`, `run_with_retry`, `get_df_info` into a FastAPI backend (`services/` module)
- Build Streamlit frontend: uploader, auto-EDA display, chat-style Q&A
- Commit incrementally to `betabytez-aiml-task5-UmmeHabiba`